# CN7 02_predict
지우 final e8e354f 원본을 사용한다. Python3.11 가상환경 활성화 후 cn7/requirements.txt 설치. 모델의 통합 이상 경보는 가스/미성형 유형 분류가 아니다. 탐지와 확률층 개발평가를 새 독립성능으로 해석하지 않는다.


In [ ]:
from pathlib import Path
import sys,os
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'cn7/vendor/jiwoo_final').exists())
VENDOR=ROOT/'cn7/vendor/jiwoo_final'
sys.path.insert(0,str(VENDOR/'src'))
import paths
paths.ROOT=ROOT
paths.DATA=ROOT/'dataset/runtime'
paths.OUT=ROOT/'cn7/outputs'
for name,rel in {'INTER':'intermediate','LINKED':'intermediate/linked','BASE':'intermediate/base','KAMP_CACHE':'intermediate/shots_cache.pkl','W_PRIOR':'intermediate/w_prior.json','FINAL':'final_model','PROB':'prob','LEDGER':'ledger','PRED':'pred','EXP':'experiments'}.items():
 setattr(paths,name,paths.OUT/rel)
paths.ensure_dirs()
os.environ['KAPPA']='10'

import pandas as pd
from IPython.display import display
import final_model as fm
# 모델을 만들지 않는다. 01에서 생성한 모델과 Run 이력이 필요하다.
INPUT_CSV=ROOT/'dataset/processed/cn7/cn7_shots.csv'
shots=pd.read_csv(INPUT_CSV,parse_dates=['TimeStamp'])
shots=shots.drop(columns=['PassOrFail','Reason'],errors='ignore')
assert shots.TimeStamp.is_unique,'샷 단위 입력 필요'
scored=fm.score_new_shots(shots,'cn7',model_dir=str(ROOT/'cn7/models'))
out=scored[['run_id','TimeStamp','score','pct','top3_local']].copy()
out['defect_alarm']=out.pct.ge(.999).astype(int)
out['inspection_candidate']=out.pct.ge(.975).astype(int)
out['predicted_type']='통합 이상 후보(가스/미성형 구분 안 함)'
out.loc[out.defect_alarm.eq(0),'predicted_type']='경보 없음(정상 확정 아님)'
outdir=ROOT/'cn7/predictions';outdir.mkdir(exist_ok=True)
out.to_csv(outdir/'cn7_predictions.csv',index=False,encoding='utf-8-sig')
print('샷',len(out),'통합 이상 경보',int(out.defect_alarm.sum()))
display(out.head())

